In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# For simple label encoding to one-hot
from sklearn.preprocessing import OneHotEncoder

from src.layers.activation import ELU
from src.layers.activation.softmax import Softmax
from src.layers.dense import Dense
from src.loss.cross_entropy import CrossEntropy
from src.models.sequential import Sequential
from src.optimizer.adam import Adam


In [5]:
from sklearn.datasets import load_digits

digits = load_digits()
x = digits.data.astype(np.float32) / 16.0 # type: ignore
y = digits.target.astype(np.int64) # type: ignore

x_train, x_test, y_train_labels, y_test_labels = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

encoder = OneHotEncoder(sparse_output=False, dtype=np.float32)
y_train = encoder.fit_transform(y_train_labels.reshape(-1, 1))
y_test = encoder.transform(y_test_labels.reshape(-1, 1))

In [ ]:


model = Sequential(
    [
        Dense(units=128, name="dense_1"),
        ELU(name="elu_1"),
        Dense(units=16, name="dense_2"),
        ELU(name="elu_2"),
        Dense(units=10, name="dense_out"),
        Softmax(name="softmax_out"),
    ]
)

model.compile(loss=CrossEntropy(), optimizer=Adam(learning_rate=1e-3))

history = model.fit(x_train, y_train, epochs=100, verbose=20)

train_probs = model.predict(x_train)
test_probs = model.predict(x_test)

train_preds = np.argmax(train_probs, axis=1)
test_preds = np.argmax(test_probs, axis=1)

train_acc = accuracy_score(y_train_labels, train_preds)
test_acc = accuracy_score(y_test_labels, test_preds)
test_precision = precision_score(y_test_labels, test_preds, average="weighted", zero_division=0)
test_recall = recall_score(y_test_labels, test_preds, average="weighted", zero_division=0)
test_f1 = f1_score(y_test_labels, test_preds, average="weighted", zero_division=0)

print(f"Final train loss: {history[-1]:.6f}")
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test precision (weighted): {test_precision:.4f}")
print(f"Test recall (weighted):    {test_recall:.4f}")
print(f"Test F1 (weighted):        {test_f1:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test_labels, test_preds))

print("\nClassification report:")
print(classification_report(y_test_labels, test_preds, zero_division=0))

sample_idx = np.arange(10)
print("Predictions:", test_preds[sample_idx])
print("Ground truth:", y_test_labels[sample_idx])

Epoch 1/100 - Loss: 2.825383
Epoch 21/100 - Loss: 1.740189
Epoch 41/100 - Loss: 1.111468
Epoch 61/100 - Loss: 0.600966
Epoch 81/100 - Loss: 0.368114
Epoch 100/100 - Loss: 0.268524
Final train loss: 0.268524
Train accuracy: 0.9506
Test accuracy:  0.9306
Test precision (weighted): 0.9311
Test recall (weighted):    0.9306
Test F1 (weighted):        0.9300

Confusion matrix:
[[35  0  1  0  0  0  0  0  0  0]
 [ 0 28  0  1  0  1  0  0  2  4]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  2 35  0  0  0  0  0  0]
 [ 0  0  0  0 36  0  0  0  0  0]
 [ 0  0  0  0  0 36  0  0  0  1]
 [ 0  1  0  0  0  0 34  0  1  0]
 [ 0  0  0  0  1  0  0 35  0  0]
 [ 0  4  2  0  0  0  0  1 28  0]
 [ 0  0  0  0  0  0  0  1  2 33]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        36
           1       0.85      0.78      0.81        36
           2       0.88      1.00      0.93        35
           3       0.97      0.95      0.96        37
